# Taller 2 — Titanic: del análisis a la predicción

**Pregunta predictiva:** dadas las características de un pasajero conocidas al embarcar, ¿con qué probabilidad
habría sobrevivido y qué tan confiable es esa estimación?

Regla de oro de este cuaderno: **todas las decisiones se toman con entrenamiento/validación**.
El conjunto de prueba se abre una sola vez, al final (Parte 16).


## 10. Formulación del problema (Pasos 39–42)

In [ ]:
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    brier_score_loss,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
    cross_validate,
    learning_curve,
)

ml = importlib.import_module('flows.05_ml_models')
sns.set_theme(style='whitegrid', context='notebook')

X, y = ml.load_dataset()
print('X:', X.shape, '| tasa de supervivencia:', round(y.mean(), 4))
X.head()

### Paso 39. Decisión previa

| Decisión | Valor | Justificación |
|---|---|---|
| Tipo | Clasificación binaria supervisada | La etiqueta es 0/1 |
| Objetivo | `Survived` | Es lo que queremos estimar |
| Momento | Al embarcar | Solo información disponible entonces |
| Métrica principal | ROC-AUC | Independiente del umbral y del desbalance (~38 % de positivos) |
| Secundarias | precisión, recall, F1, exactitud | Complementan la visión del umbral elegido |
| Costo de errores | Falso negativo más caro para evacuación; falso positivo, para asignación de recursos | Depende del uso: se documenta en `problema_ml.md` |
| Éxito | Superar la regla de género (~0,79 de exactitud, ~0,71 F1) | Un modelo complejo debe aportar algo sobre una regla simple |

La exactitud sola engaña: decir siempre "no sobrevive" ya acierta ~62 %.


### Paso 40. Auditoría de variables y fuga de datos

| Columna | ¿Existe al embarcar? | Riesgo | Decisión |
|---|---|---|---|
| `Survived`, `Survivor` | No | Fuga directa | Excluir |
| `PassengerId`, `Ticket`, `Name` | Sí | Identificadores: memorizan | Excluir (el título entra como `TTitulo`) |
| `NSex`, `NEmbarked`, `NCabin`, `NTitulo` | — | Duplican categorías con orden arbitrario | Excluir; usar *one-hot* |
| `Age` imputada en Silver | — | Estadística calculada con todos los datos | Usar `AgeOriginal` y reimputar en el Pipeline |
| `Cabin`, `TCabin`, `HasCabin` | Dudoso | Pudo registrarse mejor para supervivientes | Evaluar con y sin ella (Paso 61) |
| `Person` | Derivada de `Age`/`Sex` | Redundante | Excluir del modelo base |

`load_dataset()` ya reemplaza `Age` por `AgeOriginal`: es el primer mecanismo anti-fuga.


### Paso 41. División estratificada

In [ ]:
X_tr, X_te, y_tr, y_te = ml.split(X, y)
print('Train:', X_tr.shape, '| Test:', X_te.shape)
print('Tasa train:', round(y_tr.mean(), 4), '| Tasa test:', round(y_te.mean(), 4))
print('NO volver a usar X_te hasta la Parte 16.')

### Paso 42. Baselines

In [ ]:
cv = StratifiedKFold(ml.settings.cv_folds, shuffle=True,
                     random_state=ml.settings.random_state)

dummy_acc = cross_val_score(
    ml.build_pipeline(DummyClassifier(strategy='most_frequent')),
    X_tr, y_tr, cv=cv, scoring='accuracy')
genero_pred = (X_tr['Sex'] == 'female').astype(int)

baselines = pd.DataFrame([
    {'modelo': 'dummy (siempre "no")', 'exactitud': dummy_acc.mean(),
     'precision': np.nan, 'recall': np.nan, 'f1': np.nan},
    {'modelo': 'regla de género', 'exactitud': accuracy_score(y_tr, genero_pred),
     'precision': precision_score(y_tr, genero_pred),
     'recall': recall_score(y_tr, genero_pred),
     'f1': f1_score(y_tr, genero_pred)},
]).round(3)
display(baselines)

La regla "mujer → sobrevive" da ~0,79 de exactitud y ~0,71 de F1 sobre entrenamiento:
**esa es la vara**. Cualquier modelo debe superarla de forma honesta.


## Parte 11 — Preprocesamiento reproducible con `Pipeline` (Pasos 43–46)

### Paso 43. Demostrar la fuga de datos

In [ ]:
media_todos = X.groupby('Sex')['Age'].mean()
media_train = X_tr.groupby('Sex')['Age'].mean()
pd.DataFrame({'todos_los_datos': media_todos, 'solo_entrenamiento': media_train}).round(3)

Las medias difieren poco aquí porque los faltantes son ~20 % y los grupos son grandes; en otros datasets
(con grupos pequeños o más nulos) la diferencia puede ser grande. Aun así, usar `todos_los_datos` deja
información de `X_test` dentro del entrenamiento: es un error de método.


In [ ]:
# Paso 44: el transformador propio aprende SOLO en fit()
train_demo = pd.DataFrame({'Sex': ['male', 'male', 'female'], 'Age': [10.0, 20.0, 40.0]})
test_demo = pd.DataFrame({'Sex': ['male', 'other'], 'Age': [np.nan, np.nan]})
ml.GroupAgeImputer(group_cols=('Sex',)).fit(train_demo).transform(test_demo)

### Paso 45–46. Construir y verificar el pipeline

In [ ]:
pipe = ml.build_pipeline(ml.get_candidates(ml.settings.random_state)['logreg'])
pre = pipe[:-1].fit(X_tr)
Z = pre.transform(X_te)
print('Forma de la matriz transformada:', Z.shape)
print('¿Tiene NaN?:', bool(np.isnan(Z).any()))
print('Columnas (one-hot):', list(pre['prep'].get_feature_names_out())[:12])

`GroupAgeImputer` → mediana + escalado (numéricas) / moda + one-hot (categóricas) → modelo.
Como todo está dentro del `Pipeline`, la imputación y las categorías se aprenden solo con entrenamiento.
`Sex` aparece como `Sex_female`/`Sex_male`, sin orden implícito.


## Parte 12 — Modelos candidatos y validación cruzada (Pasos 47–50)

In [ ]:
cands = ml.get_candidates(ml.settings.random_state)
print('Candidatos:', list(cands))
cv_results = ml.cross_validate_candidates(X_tr, y_tr)
cv_results[['model', 'test_roc_auc_mean', 'test_roc_auc_std', 'test_accuracy_mean',
            'train_roc_auc_mean']].round(3)

In [ ]:
fold_rows = []
for name in ['logreg', 'forest', 'hgb']:
    res = cross_validate(ml.build_pipeline(cands[name]), X_tr, y_tr, cv=cv,
                         scoring='roc_auc', return_train_score=True)
    for i, score in enumerate(res['test_score']):
        fold_rows.append({'modelo': name, 'pliegue': i + 1, 'roc_auc': score})
folds = pd.DataFrame(fold_rows)
sns.boxplot(data=folds, x='modelo', y='roc_auc')
plt.title('Distribución de ROC-AUC por pliegue')
plt.show()

In [ ]:
gap = cv_results[['model', 'train_roc_auc_mean', 'test_roc_auc_mean']].copy()
gap['brecha_train_valid'] = gap['train_roc_auc_mean'] - gap['test_roc_auc_mean']
gap.round(3)

### Paso 50. Finalistas

- **hgb**: mejor ROC-AUC medio (~0,887), pero brecha entrenamiento–validación enorme (~0,11): memoriza.
- **forest**: ~0,866, también con sobreajuste fuerte.
- **logreg**: ~0,858, la brecha más pequeña (~0,006), interpretable, rápida y bien calibrada en el rango medio.

Elegimos **hgb** y **logreg** para profundizar, y finalmente preferimos `logreg` por estabilidad,
explicabilidad y uso previsto (informe para una aseguradora). La diferencia de AUC se discute con datos
en `reports/problema_ml.md`; la regla de oro se respeta: nada de esto mira `X_te`.


## Parte 13 — Evaluación en profundidad (Pasos 51–55)

In [ ]:
finalistas = {'logreg': cands['logreg'], 'hgb': cands['hgb'], 'forest': cands['forest']}
oof = {}
for name, est in finalistas.items():
    p = ml.build_pipeline(est)
    oof[name] = cross_val_predict(p, X_tr, y_tr, cv=cv, method='predict_proba')[:, 1]

pred_log = (oof['logreg'] >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(y_tr, pred_log)
plt.title('Matriz de confusión — logreg (fuera de muestra)')
plt.show()
print(classification_report(y_tr, pred_log, digits=3))

**Lectura para la aseguradora.** Falsos negativos: pasajeros que sobrevivieron y el modelo no los
identificó. Falsos positivos: personas a las que se daría por sobrevivientes sin serlo. Cada celda tiene un
costo distinto según el uso; por eso el umbral se decide aparte.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for name, proba in oof.items():
    RocCurveDisplay.from_predictions(y_tr, proba, name=name, ax=ax[0])
    PrecisionRecallDisplay.from_predictions(y_tr, proba, name=name, ax=ax[1])
ax[0].set_title('Curvas ROC')
ax[1].set_title('Curvas precisión–recall')
plt.show()

La curva PR es más informativa aquí: con ~38 % de positivos, la ROC puede verse optimista.


### Paso 53. Calibración y umbral

In [ ]:
proba_oof = oof['logreg']
frac_true, frac_pred = calibration_curve(y_tr, proba_oof, n_bins=8)
plt.plot([0, 1], [0, 1], 'k--', label='perfectamente calibrado')
plt.plot(frac_pred, frac_true, 'o-', label='logreg')
plt.title('Curva de calibración')
plt.xlabel('Probabilidad predicha')
plt.ylabel('Frecuencia observada')
plt.legend()
plt.show()
print('Brier:', round(brier_score_loss(y_tr, proba_oof), 4))

prec, rec, umbrales = precision_recall_curve(y_tr, proba_oof)
f1s = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
i = int(np.argmax(f1s))
print(f'Umbral que maximiza F1 en validación: {umbrales[i]:.3f} (F1={f1s[i]:.3f})')
print(f'F1 con umbral 0.5: {f1_score(y_tr, (proba_oof >= 0.5).astype(int)):.3f}')

El umbral que maximiza F1 (~0,32) es menor que 0,5. Decidimos **mantener 0,5** en el modelo congelado:
en un contexto de seguros, bajar el umbral aumenta los falsos positivos y la calibración en el rango medio
no justifica ese costo sin un análisis de costos explícito. Queda documentado como decisión conservadora.


### Paso 54. Errores por subgrupos

In [ ]:
silver = pd.read_parquet(ml.SILVER / ml.SILVER_NAME)
err = X_tr.copy()
err['y'] = y_tr.to_numpy()
err['pred'] = pred_log
err['proba'] = proba_oof
err['error'] = err['pred'] != err['y']
err['Person'] = silver.loc[err.index, 'Person'].to_numpy()

for col in ['Sex', 'Pclass', 'Person']:
    print('\n---', col, '---')
    print(err.groupby(col).agg(n=('error', 'size'), exactitud=('error', lambda s: 1 - s.mean()),
                               recall=('y', lambda s: np.nan)).round(3))
    print(err.groupby(col).apply(lambda g: pd.Series({
        'n': len(g), 'exactitud': 1 - g['error'].mean(),
        'recall': g.loc[g['y'] == 1, 'pred'].mean() if (g['y'] == 1).any() else np.nan
    }), include_groups=False).round(3))

In [ ]:
altos = err[(err['proba'] > 0.85) & err['error']]
bajos = err[(err['proba'] < 0.15) & err['error']]
errores_confianza = pd.concat([altos.nlargest(5, 'proba'), bajos.nsmallest(5, 'proba')])
errores_confianza[['Sex', 'Pclass', 'Age', 'Fare', 'TTitulo', 'y', 'pred', 'proba']]

**Reflexión ética.** `Sex` y `Pclass` reproducen las reglas de evacuación de 1912. Predecir bien lo que
ocurrió no implica que *deba* ocurrir así, ni que esas variables sean apropiadas para decidir servicios hoy.


### Paso 55. Curva de aprendizaje

In [ ]:
sizes, tr, va = learning_curve(
    ml.build_pipeline(cands['logreg']), X_tr, y_tr, cv=cv,
    train_sizes=np.linspace(0.2, 1.0, 5), scoring='roc_auc')
plt.plot(sizes, tr.mean(axis=1), 'o-', label='entrenamiento')
plt.plot(sizes, va.mean(axis=1), 'o-', label='validación')
plt.fill_between(sizes, va.mean(axis=1) - va.std(axis=1),
                 va.mean(axis=1) + va.std(axis=1), alpha=0.2)
plt.title('Curva de aprendizaje — logreg')
plt.xlabel('Tamaño de entrenamiento')
plt.ylabel('ROC-AUC')
plt.legend()
plt.show()

Las curvas se acercan a ~0,86 con poca brecha: más datos aportarían poco; el límite está en las
variables disponibles (alto sesgo), no en la varianza del modelo.


## Parte 14 — Ajuste de hiperparámetros (Pasos 56–58)

In [ ]:
grids = ml.get_param_grids()
print('Rejillas:', {k: len(v) for k, v in grids.items()})

search = {}
for name in ['logreg', 'forest', 'hgb']:
    gs = GridSearchCV(ml.build_pipeline(cands[name]), grids[name], scoring='roc_auc',
                      cv=cv, n_jobs=-1, refit=True)
    gs.fit(X_tr, y_tr)
    search[name] = gs
    print(f'\n{name}: mejores parámetros = {gs.best_params_} | CV = {gs.best_score_:.4f}')
    display(pd.DataFrame(gs.cv_results_).sort_values('rank_test_score')
            [['params', 'mean_test_score', 'std_test_score']].head(3))

In [ ]:
comparacion = []
for name, gs in search.items():
    a = cross_val_score(ml.build_pipeline(cands[name]), X_tr, y_tr, cv=cv, scoring='roc_auc')
    b = cross_val_score(gs.best_estimator_, X_tr, y_tr, cv=cv, scoring='roc_auc')
    d = b - a
    comparacion.append({'modelo': name, 'default': a.mean(), 'ajustado': b.mean(),
                        'delta': d.mean(), 'delta_std': d.std()})
pd.DataFrame(comparacion).round(4)

**Paso 58.** Para `hgb`, la mejora del ajuste (~0,006) es menor que su desviación entre pliegues
(~0,010): no está claro que sea real. Para `logreg`, C = 0,1 y C = 1 empatan (diferencia ~2e-5);
elegimos **C = 0,1** por ser la opción más regularizada y simple. Esto es lo que se congela en el módulo.


## Parte 15 — Interpretación y ablación (Pasos 59–61)

In [ ]:
modelo_lineal = ml.build_pipeline(cands['logreg']).fit(X_tr, y_tr)
nombres = modelo_lineal['prep'].get_feature_names_out()
coefs = modelo_lineal['model'].coef_[0]
coef_tbl = (pd.DataFrame({'variable': nombres, 'coeficiente': coefs,
                          'odds_ratio': np.exp(coefs)})
            .assign(abs_coef=lambda d: d['coeficiente'].abs())
            .sort_values('abs_coef', ascending=False).drop(columns='abs_coef'))
coef_tbl.round(3).head(10)

Las numéricas están estandarizadas: un coeficiente de −0,45 en `Age` significa que una desviación
típica más de edad reduce el log-odds en 0,45. Es **asociación predictiva**, no causal.


In [ ]:
X_a, X_b, y_a, y_b = ml.split(X_tr, y_tr, test_size=0.25,
                             random_state=ml.settings.random_state)
imp_model = ml.build_pipeline(cands['logreg']).fit(X_a, y_a)
pi = permutation_importance(imp_model, X_b, y_b, scoring='roc_auc', n_repeats=20,
                            random_state=ml.settings.random_state)
(pd.DataFrame({'variable': X_b.columns, 'importancia': pi.importances_mean,
               'desviacion': pi.importances_std})
 .sort_values('importancia', ascending=False).round(4))

In [ ]:
configs = {
    'base': (ml.NUM_FEATURES, ml.CAT_FEATURES),
    'base+cabina': (ml.NUM_FEATURES + ml.CABIN_NUM, ml.CAT_FEATURES + ml.CABIN_CAT),
    'base sin TTitulo': (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != 'TTitulo']),
    'base sin Sex': (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != 'Sex']),
}
abl = []
for label, (num, cat) in configs.items():
    s = cross_val_score(ml.build_pipeline(cands['logreg'], num, cat), X_tr, y_tr,
                        cv=cv, scoring='roc_auc')
    abl.append({'configuracion': label, 'roc_auc': s.mean(), 'desv': s.std()})
abl = pd.DataFrame(abl)
abl['delta_vs_base'] = abl['roc_auc'] - abl.loc[0, 'roc_auc']
abl.round(4)

### Decisión del Paso 61

Añadir cabina cambia el AUC en ~0,004, menos que la desviación entre pliegues (~0,025). Además, la cabina
pudo registrarse de forma sistemáticamente distinta para quienes sobrevivieron: es un **artefacto del registro**.
**No se incluye** en el modelo final. `TTitulo` y `Sex` tampoco se eliminan: aportan señal y su exclusión no
mejora la métrica (aunque el uso de `Sex` queda documentado como variable sensible).


## Parte 16 — Evaluación final sobre el conjunto de prueba (Pasos 62–63)

Aquí se abren `X_test`/`y_test` **por primera y única vez**, con las decisiones ya congeladas
(`FINAL_ESTIMATOR='logreg'`, `FINAL_PARAMS={'model__C': 0.1}`, sin cabina, umbral 0,5).


In [ ]:
pipe_eval = ml.train_final(X_tr, y_tr)
metricas_test = ml.evaluate_on_test(pipe_eval, X_te, y_te, ml.THRESHOLD)
proba_te = pipe_eval.predict_proba(X_te)[:, 1]
metricas_test['roc_auc_ci95'] = ml.bootstrap_ci(y_te, proba_te, roc_auc_score,
                                                random_state=ml.settings.random_state)
metricas_test['accuracy_ci95'] = ml.bootstrap_ci(
    y_te, (proba_te >= ml.THRESHOLD).astype(int), accuracy_score,
    random_state=ml.settings.random_state)
metricas_test

In [ ]:
tabla_final = pd.DataFrame([
    {'modelo': 'dummy (CV train)', 'roc_auc': 0.5, 'exactitud': dummy_acc.mean()},
    {'modelo': 'regla de género (train)', 'roc_auc': np.nan,
     'exactitud': accuracy_score(y_tr, genero_pred)},
    {'modelo': 'logreg (CV train)', 'roc_auc': cv_results.loc[
        cv_results['model'] == 'logreg', 'test_roc_auc_mean'].iloc[0],
     'exactitud': cv_results.loc[cv_results['model'] == 'logreg',
                                 'test_accuracy_mean'].iloc[0]},
    {'modelo': 'logreg (PRUEBA, una vez)', 'roc_auc': metricas_test['roc_auc'],
     'exactitud': metricas_test['accuracy']},
]).round(3)
tabla_final

### Limitaciones

- 891 pasajeros etiquetados, 179 en prueba: el intervalo bootstrap de ROC-AUC es ancho (~±0,06).
- Un solo suceso histórico; no hay validación temporal ni externa.
- 177 edades y 687 cabinas faltantes; la imputación y la categoría `X` son supuestos documentados.
- Variables sensibles (`Sex`, `Pclass`) reflejan la desigualdad de 1912: este modelo **no debe** usarse
  para decidir quién recibe un servicio hoy.
- PRUEBA se usó una sola vez: no se vuelve atrás a "arreglar" el modelo mirándola.


## Parte 17 — Entrenamiento final, guardado y prueba de humo (Pasos 64–69)

Las decisiones están congeladas en `flows/05_ml_models.py` y `make train` ya guardó el artefacto:

```python
MODEL_NAME = 'titanic_model_v1'
FINAL_ESTIMATOR = 'logreg'
FINAL_PARAMS = {'model__C': 0.1}
USE_CABIN_FEATURES = False
THRESHOLD = 0.5
```


In [ ]:
from flows.common.paths import BRONZE, BRONZE_UNLABELED_NAME, MODELS

pipe_guardado, meta = ml.load_model(MODELS / f'{ml.MODEL_NAME}.joblib')
print('Estimador:', meta['estimator'], '| parámetros:', meta['params'])
print('n_train:', meta['n_train'], '| umbral:', meta['threshold'])
print('Métricas de prueba guardadas:', meta['metrics'])

In [ ]:
de = importlib.import_module('flows.01_data_engineering')
nuevos = pd.read_csv(BRONZE / BRONZE_UNLABELED_NAME)
silver_nuevos = de.build_silver(nuevos)     # sin validate_silver: no hay etiqueta
X_nuevos = silver_nuevos[ml.CANDIDATE_COLUMNS].copy()
X_nuevos['Age'] = silver_nuevos['AgeOriginal']

predicciones = ml.predict(pipe_guardado, X_nuevos)
print('Filas predichas:', len(predicciones))
print('NaNs:', int(predicciones.isna().sum().sum()))
print('Probabilidad mín/med/max:', predicciones['proba_survive'].agg(['min', 'median', 'max']).round(3).to_dict())
print('Distribución de predicciones:', predicciones['pred'].value_counts().to_dict())
predicciones.head()

**Prueba de humo superada**: carga → transformaciones deterministas de Silver → imputación dentro del
modelo → predicción, sobre 418 pasajeros que nunca se usaron para entrenar y de los que no conocemos la
respuesta. No hay métrica que calcular; la evaluación de esos pasajeros queda para el Taller 3.

La ficha reproducible del modelo está en `reports/model_card.md`.
